In [10]:
import os

In [11]:
%pwd

'c:\\Users\\godje\\OneDrive\\Escritorio\\Data-Science-Project-Stroke-Prediction-'

In [3]:
os.chdir("../")
%pwd

'c:\\Users\\godje\\OneDrive\\Escritorio\\Data-Science-Project-Stroke-Prediction-'

In [12]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [13]:
from src.stroke_prediction.constants import *
from src.stroke_prediction.utils.common import read_yaml, create_directories

In [14]:
class ConfiguartionManager:
    def __init__(self,
                 config_file_path = CONFIG_FILE_PATH,
                 params_file_path = PARAMS_FILE_PATH,
                 schema_file_path = SCHEMA_FILE_PATH): 
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root_dir])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

In [15]:
import os
import urllib.request as request
from src.stroke_prediction import logger
import zipfile

In [16]:
#COMPONENT 1: DATA INGESTION

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self): #download the file from source url to local data file
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                self.config.source_URL,
                self.config.local_data_file
            )
            logger.info(f"File downloaded successfully: {filename}")
        else:
            logger.info(f"File already exists: {self.config.local_data_file}")


    def extract_zip_file(self):
        
        unzip_path = self.config.unzip_dir
        os.makedirs(self.config.unzip_dir, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [17]:
try:
    config = ConfiguartionManager()
    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


2026-05-06 20:11:13,811 - INFO - File downloaded successfully: artifacts/data_ingestion/dataset.zip
